In [2]:
import pandas as pd

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [14]:
SCHEMA = {
    "observed_date": "datetime64[us]",
    "departure_date": "datetime64[us]",
    "route": "string",
    "price": "float64",
    "airline":"category",
    "stops": "int8",
    "duration_min": "float32",
    "dest": "category",
    "itinerary_id": "string",
    "days_before_departure": "int16",
}

COLUMN_MAP = {
    "search_date": "observed_date",
    "destination": "dest",
    "departure_time": "departure_time",
    "duration_minutes": "duration_min",
    "price_usd": "price"
}

# Loads a dataframe with only the necessary columns
def load_flights(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df = df.rename(columns=COLUMN_MAP)

    df["departure_date"] = pd.to_datetime(df["departure_date"])
    df["observed_date"] = pd.to_datetime(df["observed_date"])

    df = df[list(SCHEMA)].astype(SCHEMA)

    return df

# validates a dataframe's columns
def validate(df: pd.DataFrame) -> dict:
    computed_days = (df["departure_date"] - df["observed_date"]).dt.days
    return {
        "rows": len(df),
        "null_count": df.isna().sum().to_dict(),
        "duplicates": int(df.duplicated().sum()),
        "price_min_max": (df["price"].min(), df["price"].max()),
        "nonpositive_prices": int((df["price"] <= 0).sum()),
        "observed_after_departure": int((df["observed_date"]> df["departure_date"]).sum()),
        "days_mismatch": int((computed_days != df["days_before_departure"]).sum()),
    }

In [16]:
def main():
    df = load_flights("data/training/sample_flights.parquet")

    df["route"].head()
    validation = validate(df)

    print(validation)

main()

{'rows': 4915392, 'null_count': {'observed_date': 0, 'departure_date': 0, 'route': 0, 'price': 0, 'airline': 0, 'stops': 0, 'duration_min': 0, 'dest': 0, 'itinerary_id': 0, 'days_before_departure': 0}, 'duplicates': 0, 'price_min_max': (np.float64(19.0), np.float64(4196.0)), 'nonpositive_prices': 0, 'observed_after_departure': 0, 'days_mismatch': 0}
